In [0]:
import json
import re


dbutils.widgets.text("run_utc", "")
dbutils.widgets.text("expected_pairs", "")
dbutils.widgets.text("expected_pending", "")
dbutils.widgets.text("expected_without_candidates", "")
dbutils.widgets.text("expected_possible_pairs", "")
dbutils.widgets.text("expected_screened_pairs", "")
run_utc = dbutils.widgets.get("run_utc")
if not re.fullmatch(r"\d{8}T\d{6}Z", run_utc):
    raise ValueError("Expected a UTC pipeline run ID")
expected_counts = {}
for key in ("expected_pairs", "expected_pending", "expected_without_candidates",
            "expected_possible_pairs", "expected_screened_pairs"):
    value = dbutils.widgets.get(key)
    if not re.fullmatch(r"\d+", value):
        raise ValueError(f"Expected a non-negative count for {key}")
    expected_counts[key] = int(value)
schema = "automation_s.og"
completed = spark.table(f"{schema}.completed_runs").where(f"run_utc = '{run_utc}'").collect()
if len(completed) != 1:
    raise ValueError(f"Expected exactly one completed run {run_utc}, found {len(completed)}")
record = completed[0]
expected = {
    "eic_project_snapshots": record["eic_projects"],
    "salesforce_opportunity_snapshots": record["salesforce_opportunities"],
    "match_review_snapshots": record["eic_projects"],
    "pair_screen_snapshots": expected_counts["expected_screened_pairs"],
}
if (record["screened_pairs"] != expected_counts["expected_screened_pairs"]
        or record["possible_pairs"] != expected_counts["expected_possible_pairs"]
        or record["possible_pairs"] != record["eic_projects"] * record["salesforce_opportunities"]):
    raise ValueError("Completed run's screening counts are inconsistent")
for table, count in expected.items():
    actual = spark.table(f"{schema}.{table}").where(f"run_utc = '{run_utc}'").count()
    if actual != count:
        raise ValueError(f"{table} row count {actual} != {count}")
for view, count in (("project_overview_current", record["eic_projects"]),
                    ("salesforce_opportunities_current", record["salesforce_opportunities"])):
    frame = spark.table(f"{schema}.{view}")
    if frame.where(f"run_utc = '{run_utc}'").count() != count or frame.count() != count:
        raise ValueError(f"{view} does not represent completed run {run_utc}")
pairs = spark.table(f"{schema}.pair_review_current")
if pairs.where(f"run_utc = '{run_utc}'").count() != expected_counts["expected_pairs"] or pairs.count() != expected_counts["expected_pairs"]:
    raise ValueError("Pair view count differs from refresh summary")
pending = pairs.where("pair_decision = 'pending'").count()
if pending != expected_counts["expected_pending"]:
    raise ValueError(f"Pair view pending count {pending} differs from refresh summary")
screening = spark.table(f"{schema}.pair_screen_current")
if (screening.count() != expected_counts["expected_possible_pairs"]
        or screening.where("screen_reason = 'candidate'").count() != expected_counts["expected_pairs"]):
    raise ValueError("Screened pair coverage or candidate decisions differ from refresh summary")
without_candidates = spark.table(f"{schema}.match_review_current").where(
    "size(candidate_ids) = 0").count()
if without_candidates != expected_counts["expected_without_candidates"]:
    raise ValueError("Projects without candidates differ from refresh summary")
subtypes = {row["project_subtype"]: row["count"] for row in spark.table(
    f"{schema}.salesforce_opportunities_current").groupBy("project_subtype").count().collect()}
if subtypes != {"Oil&Gas": record["oil_gas_opportunities"], "Gas": record["gas_opportunities"]}:
    raise ValueError(f"Salesforce subtypes differ from completed run: {subtypes}")
quality = spark.table(f"{schema}.pair_screen_quality_current").first()
result = {"status": "validated", "run_utc": run_utc, "eic_projects": record["eic_projects"],
          "salesforce_opportunities": record["salesforce_opportunities"], "subtypes": subtypes,
          "candidate_pairs": expected_counts["expected_pairs"], "pending_pairs": pending,
          "possible_pairs": expected_counts["expected_possible_pairs"],
          "screened_pairs": expected_counts["expected_screened_pairs"],
          "screen_reasons": {row["screen_reason"]: row["count"] for row in
                             screening.groupBy("screen_reason").count().collect()},
          "labelled_positive_pairs": quality["labelled_positive_pairs"],
          "labelled_retrieval_recall": quality["labelled_retrieval_recall"],
          "projects_without_candidates": without_candidates, "llm_calls": record["llm_calls"]}
print(json.dumps(result, sort_keys=True))
dbutils.notebook.exit(json.dumps(result))